In [78]:
import os
import json
import torch
from datetime import datetime, timezone

CONSISTENCY_BLACKBOARD_PATH = "/kaggle/input/notebooks/divyanshukj4495/consistency-checker/consistency_blackboard.pt"
TRIAGE_BLACKBOARD_PATH = "/kaggle/working/triage_blackboard.pt"

print("Consistency blackboard exists:", os.path.exists(CONSISTENCY_BLACKBOARD_PATH))

Consistency blackboard exists: True


In [79]:
def load_consistency_blackboard(path=CONSISTENCY_BLACKBOARD_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Consistency blackboard not found: {path}. "
            "Run the Consistency Checker Agent first."
        )

    blackboard = torch.load(path, map_location="cpu", weights_only=False)

    if not isinstance(blackboard, dict):
        raise TypeError("Blackboard must be a dictionary.")
    if "output" not in blackboard:
        raise KeyError("Blackboard must contain 'output'.")
    if not isinstance(blackboard["output"], dict):
        raise TypeError("blackboard['output'] must be a dictionary.")

    return blackboard

if os.path.exists(CONSISTENCY_BLACKBOARD_PATH):
    consistency_blackboard = load_consistency_blackboard()
    consistency_report = consistency_blackboard["output"]
    print("Loaded Consistency Checker output.")
else:
    consistency_blackboard = None
    consistency_report = None
    print("Run the previous stage before executing the production pipeline.")

Loaded Consistency Checker output.


In [80]:
GRADE_LABELS = {
    0: "No DR",
    1: "Mild NPDR",
    2: "Moderate NPDR",
    3: "Severe NPDR",
    4: "PDR",
}

BASE_TIER_BY_GRADE = {
    0: "routine",
    1: "routine",
    2: "soon",
    3: "urgent",
    4: "urgent",
}

TIER_LABELS = {
    "routine": "ROUTINE",
    "soon": "SOON",
    "urgent": "URGENT",
    "emergency": "EMERGENCY",
    "clinician_review": "CLINICIAN_REVIEW",
}

In [81]:
def _flatten_values(value):
    if value is None:
        return []
    if isinstance(value, dict):
        out = []
        for k, v in value.items():
            out.append(str(k))
            out.extend(_flatten_values(v))
        return out
    if isinstance(value, (list, tuple, set)):
        out = []
        for item in value:
            out.extend(_flatten_values(item))
        return out
    return [str(value)]

def detect_urgency_cues(clinical_context=None):
    text = " | ".join(v.strip().lower() for v in _flatten_values(clinical_context))

    return {
        "vitreous_hemorrhage": (
            "vitreous hemorrhage" in text or
            "vitreous haemorrhage" in text
        ),
        "sudden_vision_loss": (
            "sudden vision loss" in text or
            "sudden visual loss" in text
        ),
        "rubeosis": "rubeosis" in text,
    }

In [82]:
def get_tta_status(report):
    grades = report.get("tta_grades") or []

    if len(grades) >= 2:
        try:
            spread = max(int(x) for x in grades) - min(int(x) for x in grades)
        except (TypeError, ValueError):
            spread = report.get("tta_spread")
    else:
        spread = report.get("tta_spread")

    if spread is None:
        return {
            "tta_spread": None,
            "tta_consistent": report.get("tta_consistent"),
            "review_required": bool(report.get("review_required", False)),
        }

    return {
        "tta_spread": int(spread),
        "tta_consistent": int(spread) <= 1,
        "review_required": int(spread) > 1,
    }

def base_tier_from_grade(grade):
    grade = int(grade)
    if grade not in BASE_TIER_BY_GRADE:
        raise ValueError("DR grade must be between 0 and 4.")
    return BASE_TIER_BY_GRADE[grade]

In [83]:
def refine_for_access(base_tier, access_context=None):
    if not access_context:
        return base_tier, "No access-context adjustment applied."

    unavailable = access_context.get("specialist_available") is False

    try:
        long_wait = (
            access_context.get("expected_wait_days") is not None
            and float(access_context["expected_wait_days"]) > 14
        )
    except (TypeError, ValueError):
        long_wait = False

    constrained = unavailable or long_wait

    if not constrained:
        return base_tier, "No access-context tier adjustment applied."

    if base_tier == "routine":
        return "soon", "Access constraint increased routine tier to soon."

    if base_tier == "soon":
        return "urgent", "Access constraint increased soon tier to urgent."

    return base_tier, "Access constraint did not change the urgent tier."

In [84]:
def triage_agent(
    consistency_report,
    clinical_context=None,
    access_context=None,
):
    if not isinstance(consistency_report, dict):
        raise TypeError("consistency_report must be a dictionary.")

    if "grade" not in consistency_report:
        raise KeyError("consistency_report must contain 'grade'.")

    grade = int(consistency_report["grade"])
    if grade not in GRADE_LABELS:
        raise ValueError("grade must be between 0 and 4.")

    grade_label = consistency_report.get("grade_label", GRADE_LABELS[grade])
    confidence = consistency_report.get("confidence")

    urgency_cues = detect_urgency_cues(clinical_context)
    emergency_cues = [k for k, v in urgency_cues.items() if v]

    tta = get_tta_status(consistency_report)
    base_tier = base_tier_from_grade(grade)
    access_tier, access_reason = refine_for_access(
        base_tier, access_context
    )

    # Highest-priority routing rules.
    if emergency_cues:
        referral_tier = "emergency"
        review_required = True
        review_reason = "Emergency cue detected: " + ", ".join(emergency_cues)

    elif tta["review_required"]:
        referral_tier = "clinician_review"
        review_required = True
        review_reason = (
            f"TTA grade spread is {tta['tta_spread']}; "
            "spread > 1 requires clinician review."
        )

    elif consistency_report.get("review_required", False):
        referral_tier = "clinician_review"
        review_required = True
        review_reason = consistency_report.get(
            "review_reason",
            "Consistency Checker requested clinician review."
        )

    else:
        referral_tier = access_tier
        review_required = False
        review_reason = None

    return {
        "agent": "TriageAgent",
        "grade": grade,
        "grade_label": grade_label,
        "confidence": confidence,

        "base_referral_tier": base_tier,
        "access_adjusted_tier": access_tier,
        "referral_tier": referral_tier,
        "referral_tier_label": TIER_LABELS[referral_tier],

        "urgency_cues": urgency_cues,
        "emergency_cues_detected": emergency_cues,

        "tta_spread": tta["tta_spread"],
        "tta_consistent": tta["tta_consistent"],

        "clinical_completeness": consistency_report.get(
            "clinical_completeness"
        ),
        "clinical_missing_fields": consistency_report.get(
            "clinical_missing_fields", []
        ),

        "access_context_used": access_context is not None,
        "access_adjustment_reason": access_reason,

        "review_required": review_required,
        "review_reason": review_reason,

        "triage_status": "ESCALATE" if review_required else "ROUTED",
    }

In [85]:
if consistency_report is not None:
    # Replace None with the actual Clinical-Note Agent output.
    clinical_context = None

    triage_output = triage_agent(
        consistency_report=consistency_report,
        clinical_context=clinical_context,
        access_context=None,
    )

    print(json.dumps(triage_output, indent=2, default=str))
else:
    triage_output = None
    print("Consistency blackboard not found. Run the previous agent first.")

{
  "agent": "TriageAgent",
  "grade": 0,
  "grade_label": "No DR",
  "confidence": 0.36490851640701294,
  "base_referral_tier": "routine",
  "access_adjusted_tier": "routine",
  "referral_tier": "routine",
  "referral_tier_label": "ROUTINE",
  "urgency_cues": {
    "vitreous_hemorrhage": false,
    "sudden_vision_loss": false,
    "rubeosis": false
  },
  "emergency_cues_detected": [],
  "tta_spread": 0,
  "tta_consistent": true,
  "clinical_completeness": "LARGELY_MISSING",
  "clinical_missing_fields": [
    "visual_acuity",
    "lens_status",
    "prior_laser",
    "prior_anti_vegf",
    "hba1c",
    "prior_vitrectomy",
    "symptoms"
  ],
  "access_context_used": false,
  "access_adjustment_reason": "No access-context adjustment applied.",
  "review_required": false,
  "review_reason": null,
  "triage_status": "ROUTED"
}


In [86]:
def test_report(grade, tta_grades=None, confidence=0.84):
    tta_grades = tta_grades or [grade, grade, grade]
    spread = max(tta_grades) - min(tta_grades)

    return {
        "grade": grade,
        "grade_label": GRADE_LABELS[grade],
        "confidence": confidence,
        "tta_grades": tta_grades,
        "tta_spread": spread,
        "tta_consistent": spread <= 1,
        "clinical_missing_fields": [],
        "clinical_completeness": "COMPLETE",
        "review_required": False,
        "review_reason": None,
    }

tests = [
    ("Grade 0", test_report(0), None, "routine"),
    ("Grade 1", test_report(1), None, "routine"),
    ("Grade 2", test_report(2), None, "soon"),
    ("Grade 3", test_report(3), None, "urgent"),
    ("Grade 4", test_report(4), None, "urgent"),
    ("Emergency", test_report(2), {"symptoms": ["sudden vision loss"]}, "emergency"),
    ("TTA review", test_report(2, [1, 2, 3]), None, "clinician_review"),
]

for name, report, context, expected in tests:
    result = triage_agent(report, context, None)
    assert result["referral_tier"] == expected, (
        f"{name}: expected {expected}, got {result['referral_tier']}"
    )
    print(f"{name:15s} -> {result['referral_tier']} ✓")

print("All Triage Agent tests passed.")

Grade 0         -> routine ✓
Grade 1         -> routine ✓
Grade 2         -> soon ✓
Grade 3         -> urgent ✓
Grade 4         -> urgent ✓
Emergency       -> emergency ✓
TTA review      -> clinician_review ✓
All Triage Agent tests passed.


In [87]:
if triage_output is not None:
    triage_blackboard = {
        "agent": "TriageAgent",
        "input_agent": "ConsistencyCheckerAgent",
        "output": triage_output,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

    torch.save(triage_blackboard, TRIAGE_BLACKBOARD_PATH)

    print("Saved:", TRIAGE_BLACKBOARD_PATH)
    print("Exists:", os.path.exists(TRIAGE_BLACKBOARD_PATH))
else:
    print("No production blackboard saved because the Consistency Checker output is unavailable.")

Saved: /kaggle/working/triage_blackboard.pt
Exists: True
